In [1]:
import os
import requests
import json
from datetime import datetime
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

In [2]:
# Coordenadas de Posadas, Misiones
LATITUD = -27.3918
LONGITUD = -55.9238

# URL de la API de Open-Meteo
URL = "https://api.open-meteo.com/v1/forecast?latitude=-27.3918&longitude=-55.9238&hourly=temperature_2m,relative_humidity_2m,rain,precipitation"

# Parámetros de la consulta
params = {
    "latitude": LATITUD,
    "longitude": LONGITUD,
    "hourly": ["temperature_2m", "relative_humidity_2m", "rain", "precipitation"],
    "timezone": "America/Argentina/Buenos_Aires"
}

In [4]:
# 1. Petición a la API
response = requests.get(URL, params=params)

if response.status_code == 200:
    data_raw = response.json()
    
    # CONTROL DE SEGURIDAD: Open-Meteo a veces devuelve una lista si mandas ciertos parámetros
    if isinstance(data_raw, list):
        # Si es una lista, tomamos el primer elemento que contiene los datos de la locación
        data_raw = data_raw[0]
    
    # Verificamos si la clave 'hourly' realmente existe en el diccionario
    if "hourly" in data_raw:
        # 2. Convertimos la sección 'hourly' en un DataFrame crudo
        df_bronze = pd.DataFrame(data_raw["hourly"])
        
        # 3. Metadato de control: ¿Cuándo se descargaron estos datos?
        df_bronze["ingested_at"] = datetime.now()
        
        print(f"¡Datos extraídos con éxito! Filas obtenidas: {len(df_bronze)}")
    else:
        print("Error: La clave 'hourly' no se encuentra en la respuesta.")
        print("Estructura recibida:", data_raw.keys())
else:
    print(f"Error al conectar con la API: {response.status_code}")

¡Datos extraídos con éxito! Filas obtenidas: 168


In [5]:
# Definir la ruta de guardado
fecha_hoy = datetime.now().strftime("%Y%m%d_%H%M")
ruta_destino = f"../Data/Bronze/weather_raw_{fecha_hoy}.parquet"

# Asegurar que la carpeta exista
os.makedirs("../Data/Bronze/", exist_ok=True)

# Guardar en formato Parquet usando PyArrow
table = pa.Table.from_pandas(df_bronze)
pq.write_table(table, ruta_destino)

print(f"Datos crudos guardados exitosamente en: {ruta_destino}")

Datos crudos guardados exitosamente en: ../Data/Bronze/weather_raw_20260625_1208.parquet
